# FleetLogix — Avance 1: Generación y control de calidad de datos

**Proyecto Integrador Módulo 2 · Data Science**  
**Estudiante:** Lucas, Francisco Magrino  
**Tecnologías:** PostgreSQL 18, Python 3, Faker, Pandas y NumPy.

## Objetivo
Analizar un modelo relacional de logística, generar y cargar **505.650 registros sintéticos** en seis tablas interrelacionadas, y demostrar su integridad y coherencia mediante verificaciones reproducibles.

> **Importante:** este notebook **no vuelve a cargar** datos en PostgreSQL ni ejecuta automáticamente el generador. Documenta el procedimiento y analiza los archivos CSV ya creados. El generador entregable se encuentra en `01_data_generation.py`.

## 1. Modelo relacional y reglas de negocio

| Tabla | Clave primaria | Relaciones | Registros esperados |
|---|---|---|---:|
| `vehicles` | `vehicle_id` | Referenciada por viajes y mantenimientos | 200 |
| `drivers` | `driver_id` | Referenciada por viajes | 400 |
| `routes` | `route_id` | Referenciada por viajes | 50 |
| `trips` | `trip_id` | FK a vehículos, conductores y rutas | 100.000 |
| `deliveries` | `delivery_id` | FK a viajes | 400.000 |
| `maintenance` | `maintenance_id` | FK a vehículos | 5.000 |

Se utilizaron **cinco relaciones de clave foránea**, campos obligatorios y restricciones de unicidad definidas en `fleetlogix_db_schema.sql`. La estructura original de las seis tablas se conservó.

**Interpretación de negocio:** un vehículo puede participar en varios viajes y tener múltiples mantenimientos; un conductor puede realizar varios viajes; una ruta puede reutilizarse; cada viaje puede tener varias entregas. Los viajes históricos cubren el periodo **01/10/2024 al 30/09/2026** y se buscó un promedio de cuatro entregas por viaje.

**Diagrama

## 3. Localización y lectura de archivos CSV

Colocá la carpeta `fleetlogix_datos` **junto al notebook**. Si está en otra carpeta, cambiá `CARPETA_DATOS` por su ruta real. No es necesario ejecutar otra vez el generador.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


CARPETA_DATOS = Path(r"C:\Users\magri\OneDrive\Desktop\SQL PI\fleetlogix_datos")
NOMBRES = ["vehicles", "drivers", "routes", "trips", "deliveries", "maintenance"]
ESPERADOS = dict(zip(NOMBRES, [200, 400, 50, 100000, 400000, 5000]))

if not CARPETA_DATOS.exists():
    raise FileNotFoundError(
        "No encuentro la carpeta 'fleetlogix_datos'. "
        "Copiala al lado de este notebook o ajustá CARPETA_DATOS a su ruta."
    )

faltantes = [f"{nombre}.csv" for nombre in NOMBRES if not (CARPETA_DATOS / f"{nombre}.csv").exists()]
if faltantes:
    raise FileNotFoundError(f"Faltan los CSV: {faltantes}")

tablas = {nombre: pd.read_csv(CARPETA_DATOS / f"{nombre}.csv") for nombre in NOMBRES}
print("Archivos cargados correctamente:", ", ".join(NOMBRES))

Archivos cargados correctamente: vehicles, drivers, routes, trips, deliveries, maintenance


## 4. Comprobación de cantidades

In [ ]:
resumen = pd.DataFrame([
    {"tabla": nombre, "esperados": ESPERADOS[nombre], "obtenidos": len(tablas[nombre]),
     "coincide": len(tablas[nombre]) == ESPERADOS[nombre]}
    for nombre in NOMBRES
])
display(resumen)
print("TOTAL:", resumen["obtenidos"].sum())
assert resumen["coincide"].all(), "Alguna cantidad no coincide"
assert resumen["obtenidos"].sum() == 505650
print("✅ Cantidades correctas")

## 5. Integridad referencial, unicidad y fechas

En el esquema relacional, una FK debe referenciar una PK existente. Las siguientes comprobaciones también detectan referencias vacías, aunque las columnas FK originales no tenían todas `NOT NULL`.

In [ ]:
v, d, r, t, e, m = (tablas[n] for n in NOMBRES)
relaciones = [
    ("trips → vehicles", t, "vehicle_id", v, "vehicle_id"),
    ("trips → drivers", t, "driver_id", d, "driver_id"),
    ("trips → routes", t, "route_id", r, "route_id"),
    ("deliveries → trips", e, "trip_id", t, "trip_id"),
    ("maintenance → vehicles", m, "vehicle_id", v, "vehicle_id"),
]
reporte_fk = pd.DataFrame([
    {"relacion": nombre, "errores": int((~hija[clave_hija].isin(padre[clave_padre])).sum())}
    for nombre, hija, clave_hija, padre, clave_padre in relaciones
])
display(reporte_fk)
assert (reporte_fk["errores"] == 0).all(), "Hay referencias inválidas o vacías"
print("✅ Integridad referencial correcta")

In [ ]:
restricciones_unicas = {
    "vehicles": ["vehicle_id", "license_plate"],
    "drivers": ["driver_id", "employee_code", "license_number"],
    "routes": ["route_id", "route_code"],
    "trips": ["trip_id"],
    "deliveries": ["delivery_id", "tracking_number"],
    "maintenance": ["maintenance_id"],
}
repetidos = [
    {"tabla": nombre, "columna": columna,
     "duplicados": int(tablas[nombre][columna].duplicated().sum())}
    for nombre, columnas in restricciones_unicas.items() for columna in columnas
]
display(pd.DataFrame(repetidos))
assert all(fila["duplicados"] == 0 for fila in repetidos)
print("✅ Claves y códigos únicos")

In [ ]:
t = t.copy()
e = e.copy()
for c in ["departure_datetime", "arrival_datetime"]:
    t[c] = pd.to_datetime(t[c], errors="coerce")
for c in ["scheduled_datetime", "delivered_datetime"]:
    e[c] = pd.to_datetime(e[c], errors="coerce")

viajes_fechas_invalidas = int((
    t["departure_datetime"].isna() |
    t["arrival_datetime"].isna() |
    (t["arrival_datetime"] <= t["departure_datetime"])
).sum())

ventanas = t.set_index("trip_id")[["departure_datetime", "arrival_datetime"]]
rel_ent = e[["trip_id", "scheduled_datetime", "delivered_datetime"]].join(ventanas, on="trip_id")
entregas_fechas_invalidas = int((
    rel_ent["scheduled_datetime"].isna() |
    rel_ent["delivered_datetime"].isna() |
    (rel_ent["delivered_datetime"] < rel_ent["scheduled_datetime"]) |
    (rel_ent["delivered_datetime"] > rel_ent["arrival_datetime"]) |
    (rel_ent["delivered_datetime"] < rel_ent["departure_datetime"])
).sum())

print("Viajes con fechas inválidas:", viajes_fechas_invalidas)
print("Entregas con fechas inválidas:", entregas_fechas_invalidas)
assert viajes_fechas_invalidas == 0
assert entregas_fechas_invalidas == 0
print("✅ Validación temporal correcta")

## 6. Distribuciones y reglas de negocio

In [ ]:
entregas_por_viaje = e.groupby("trip_id").size().reindex(t["trip_id"], fill_value=0)
print("Promedio entregas/viaje:", round(entregas_por_viaje.mean(), 2))
print("Mínimo:", entregas_por_viaje.min(), "| Máximo:", entregas_por_viaje.max())
print("Viajes con menos de 2 o más de 6 entregas:", int((~entregas_por_viaje.between(2, 6)).sum()))
print("Mantenimientos / viajes:", f"1 cada {len(t)/len(m):.1f} viajes")
assert entregas_por_viaje.between(2, 6).all()
assert abs(entregas_por_viaje.mean() - 4) < 0.01
print("✅ Proporciones de entregas y mantenimiento coherentes")

In [ ]:
resumen_horas = (t.assign(hora=t["departure_datetime"].dt.hour)
                 .groupby("hora").size().reindex(range(24), fill_value=0)
                 .rename("viajes"))
resumen_meses = (t.assign(mes=t["departure_datetime"].dt.to_period("M").astype(str))
                 .groupby("mes").size().rename("viajes"))
print("Distribución por hora:")
display(resumen_horas.to_frame().T)
print("Distribución por mes:")
display(resumen_meses.to_frame())
print("Tipos de vehículos:")
display(v["vehicle_type"].value_counts().rename("vehiculos").to_frame())
print("Tipos de mantenimiento:")
display(m["maintenance_type"].value_counts().rename("mantenimientos").to_frame())

print("Periodo observado:", t["departure_datetime"].min(), "a", t["departure_datetime"].max())
print("Viajes completados (%):", round(100 * t["status"].eq("completed").mean(), 1))
print("Entregas completadas (%):", round(100 * e["delivery_status"].eq("delivered").mean(), 1))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
resumen_horas.plot(kind="bar", ax=ax)
ax.set(title="FleetLogix — viajes por hora del día", xlabel="Hora de salida", ylabel="Cantidad de viajes")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
resumen_meses.plot(kind="bar", ax=ax)
ax.set(title="FleetLogix — viajes por mes", xlabel="Mes", ylabel="Cantidad de viajes")
plt.tight_layout()
plt.show()

## 7. Capacidad de vehículos y mejora de calidad

Se comprueba que `trips.total_weight_kg` no supere la capacidad registrada en `vehicles`. También se presenta una verificación adicional del **peso agregado de los paquetes** frente al peso declarado del viaje. Esta última sirve para detectar posibles límites del generador; si da valores positivos, hay que **documentar y corregir el modelo sintético**, no afirmar que cumple todo.

In [ ]:
capacidades = v.set_index("vehicle_id")["capacity_kg"]
viaje_cap = t.join(capacidades, on="vehicle_id")
sobrecarga = int((
    viaje_cap["total_weight_kg"].isna() |
    (viaje_cap["total_weight_kg"] < 0) |
    (viaje_cap["total_weight_kg"] > viaje_cap["capacity_kg"])
).sum())

paquetes_kg = e.groupby("trip_id")["package_weight_kg"].sum()
comparacion_pesos = t.set_index("trip_id")[["total_weight_kg"]].join(paquetes_kg.rename("kg_paquetes"))
diferencias_peso = int((comparacion_pesos["kg_paquetes"] > comparacion_pesos["total_weight_kg"] + 1e-8).sum())

print("Viajes con sobrecarga sobre capacidad de vehículo:", sobrecarga)
print("Viajes con suma de paquetes mayor al peso declarado:", diferencias_peso)
if diferencias_peso:
    print("⚠️ Hallazgo a revisar: la suma del peso de paquetes supera al total de algunos viajes.")
else:
    print("✅ Los pesos agregados de paquetes no superan el total declarado por viaje.")

## 8. Evidencias de carga a PostgreSQL

Se importaron los CSV desde **pgAdmin 4** en orden compatible con las claves foráneas: `vehicles → drivers → routes → trips → deliveries → maintenance`.

Las consultas SQL guardadas en el proyecto permiten verificar:

- Las seis tablas y sus restricciones (`01` a `05`).
- La carga total de **505.650 registros**.
- La integridad referencial (`08_validacion_integridad.sql`): cinco verificaciones con **0 referencias inválidas**.
- Las fechas de viajes y entregas (`09_validacion_fechas.sql` y consultas relacionadas): **0 inconsistencias** según las condiciones utilizadas.
- El promedio de entregas por viaje (`11_validacion_distribucion_entregas.sql`): **4,00**, mínimo **2**, máximo **6**.
- Distribuciones (`13_distribuciones_estadisticas.sql`): predominio de salidas en horario diurno y presencia de registros mensuales en el período de operación.

**Evidencia visual pendiente de adjuntar:** capturas del conteo de tablas, resultados de validación y diagrama ER. Los resultados indicados provienen de las comprobaciones realizadas en pgAdmin; el notebook permite repetir sus equivalentes sobre CSV.

## 9. Posibles mejoras sin alterar el esquema original

1. **Control de disponibilidad:** evitar asignar el mismo conductor o vehículo a viajes simultáneos (actualmente no se valida exhaustivamente).
2. **Mayor realismo en estados:** introducir entregas fallidas, retrasadas o pendientes, de acuerdo con reglas de negocio definidas.
3. **Consistencia de pesos por viaje:** asegurar que los pesos individuales de paquetes sean compatibles con el peso declarado del viaje.
4. **Coherencia geográfica:** usar distancias y duraciones acordes a los pares de ciudades y distinguir rutas urbanas de interurbanas.
5. **Trazabilidad:** conservar versión de librerías, log de ejecución, resultados de validación y evidencia SQL en la entrega.


## 10. Funcionamiento del generador de datos

El archivo `01_data_generation.py` utiliza Faker, Pandas y NumPy para generar información sintética de las seis tablas de FleetLogix.

**Función `generate_trips()`:**

Genera 100.000 viajes distribuidos durante dos años. Asigna identificadores válidos de vehículos, conductores y rutas. Calcula las fechas de salida y llegada, el combustible consumido y el peso transportado.

**Función `gethourly_distribution()`:**

Define una distribución de probabilidades para las horas de salida de los viajes. Se asigna mayor probabilidad a los horarios laborales y menor probabilidad a la madrugada.

**Función `generate_deliveries()`:**

Genera 400.000 entregas relacionadas con viajes existentes, manteniendo un promedio de cuatro entregas por viaje.

**Función `generate_maintenance()`:**

Genera 5.000 registros de mantenimiento asociados a vehículos existentes.

**Validaciones:**

El generador comprueba las cantidades de registros, las claves foráneas, las fechas y la unicidad de los identificadores correspondientes.


## 11. Conclusión

Se construyó una base sintética de **505.650 registros** en seis tablas relacionadas. La población de PostgreSQL y las verificaciones de cantidades, relaciones y fechas fueron comprobadas durante el trabajo. El análisis de distribuciones muestra patrones horarios de actividad coherentes con una operación diurna. Se identifican oportunidades para fortalecer el realismo de estados, las restricciones operativas y la consistencia de pesos antes de considerar el dataset una simulación exhaustiva de operaciones reales.

**Archivos a entregar para el Avance 1:** `Avance_1_DataGeneration.ipynb`, `01_data_generation.py`, esquema SQL y diagrama ER como evidencia complementaria. Seguir las indicaciones de la universidad para la carpeta final.